# Laboratório 5 — Câmera Estéreo

**Disciplina:** ESZA019 — Visão Computacional — UFABC 2026.2
**Professor:** Celso Setsuo Kurashima
**Equipe:** Ctrl+C, Ctrl+V e Fé

### Integrantes da equipe
- Lucas Rodrigues Teixeira — RA 11202131394
- Pedro Henrique Garcez Silva — RA 11202130642
- Roberto Sene Azevedo — RA 11202020360

**Data de realização dos experimentos:** 13/07/2026
**Data de publicação do relatório:** 15/07/2026


## Sumário

1. [Introdução](#1-introdução)
2. [Fundamentação teórica (Parte 1)](#2-fundamentação-teórica-parte-1)
   - 2.1 [Estereoscopia e visão 3D](#21-estereoscopia-e-visão-3d)
   - 2.2 [Geometria epipolar: epipolos, plano e linha epipolar](#22-geometria-epipolar-epipolos-plano-e-linha-epipolar)
   - 2.3 [Matriz fundamental e matriz essencial](#23-matriz-fundamental-e-matriz-essencial)
   - 2.4 [Disparidade estéreo](#24-disparidade-estéreo)
   - 2.5 [Calibração estéreo e retificação](#25-calibração-estéreo-e-retificação)
3. [Construção da câmera estéreo (Parte 2)](#3-construção-da-câmera-estéreo-parte-2)
4. [Ambiente e instruções de reprodução](#4-ambiente-e-instruções-de-reprodução)
5. [Procedimentos experimentais (Parte 3)](#5-procedimentos-experimentais-parte-3)
   - 5.1 [(A) Obtenção dos códigos de exemplo](#51-a-obtenção-dos-códigos-de-exemplo)
   - 5.2 [(B) Execução do exemplo com as imagens fornecidas](#52-b-execução-do-exemplo-com-as-imagens-fornecidas)
   - 5.3 [(C) Calibração da câmera estéreo construída](#53-c-calibração-da-câmera-estéreo-construída)
   - 5.4 [(D) Gravação de um vídeo 3D anaglifo](#54-d-gravação-de-um-vídeo-3d-anaglifo)
   - 5.5 [(E) Aplicação no Trabalho Final](#55-e-aplicação-no-trabalho-final)
6. [Análise e discussão](#6-análise-e-discussão)
7. [Conclusões](#7-conclusões)
8. [Declaração de uso de Inteligência Artificial Generativa](#8-declaração-de-uso-de-inteligência-artificial-generativa)
9. [Referências](#9-referências)


## 1. Introdução

Nos laboratórios anteriores a equipe estudou a captura de imagens (Lab 1), *features* e correspondência
(Lab 2), homografia e mosaico (Lab 3) e a **calibração de uma única câmera** (Lab 4). Neste Laboratório
5 damos o passo natural seguinte: usar **duas câmeras** para recuperar a **profundidade** de uma cena —
a **visão estéreo**.

O trabalho consistiu em (i) estudar a **geometria epipolar** que relaciona duas vistas de uma mesma
cena; (ii) **construir** fisicamente uma câmera estereoscópica de baixo custo com duas webcams iguais;
(iii) **calibrá-la** com um padrão de tabuleiro de xadrez; e (iv) gerar, a partir das imagens
retificadas, um **vídeo 3D anaglifo** (para óculos vermelho-ciano). Este relatório é **didático e
autossuficiente** e reúne a teoria, o procedimento de construção, os programas de captura, calibração e
exibição/gravação 3D, e os **resultados efetivamente obtidos** pela equipe.


## 2. Fundamentação teórica (Parte 1)

> Esta seção também responde às perguntas teóricas da Parte 1 do roteiro (epipolos, plano epipolar, linha
> epipolar, parâmetros da matriz fundamental e disparidade estéreo).


### 2.1 Estereoscopia e visão 3D

A **estereoscopia** é o princípio pelo qual dois pontos de vista ligeiramente diferentes de uma mesma
cena permitem recuperar a **profundidade** — exatamente como os dois olhos humanos. Cada olho (ou câmera)
vê o mundo de uma posição distinta; o cérebro (ou o algoritmo) compara as duas imagens e, a partir do
**deslocamento** dos objetos entre elas, infere a que distância cada um está. Objetos próximos deslocam-se
muito entre as duas vistas; objetos distantes, pouco.

A distância entre os dois centros ópticos chama-se **baseline**. Para imitar a visão humana, o roteiro
pede que a baseline seja aproximadamente igual à **distância interpupilar** de um integrante da equipe.


### 2.2 Geometria epipolar: epipolos, plano e linha epipolar

A **geometria epipolar** é a geometria projetiva intrínseca a **duas vistas**. Considere um ponto 3D $X$
observado pelas duas câmeras, com centros ópticos $O_L$ e $O_R$:

- **Plano epipolar:** o plano que contém o ponto $X$ e os dois centros ópticos $O_L$ e $O_R$. Todo ponto
  da cena define um plano epipolar (todos passam pela reta $O_L O_R$, a *baseline*).
- **Epipolos ($e_L$, $e_R$):** o ponto onde a reta que liga os dois centros ópticos (a *baseline*)
  fura cada plano de imagem. Equivalentemente, o epipolo de uma câmera é a **imagem do centro óptico da
  outra câmera**. Todas as linhas epipolares de uma imagem passam pelo seu epipolo.
- **Linha epipolar:** a interseção do plano epipolar com o plano de imagem. Sua importância é prática: o
  correspondente de um ponto $x_L$ da imagem esquerda **necessariamente** está sobre a linha epipolar
  correspondente na imagem direita (a **restrição epipolar**). Isso reduz a busca de correspondência de
  um problema 2D para um problema **1D** ao longo de uma reta.


### 2.3 Matriz fundamental e matriz essencial

A restrição epipolar é codificada por uma matriz $3\times3$. Para pontos em coordenadas de **pixel**
$x_L$ e $x_R$, vale

$$x_R^{\top}\, F\, x_L = 0,$$

onde $F$ é a **matriz fundamental**. Suas características:

- É $3\times3$ e tem **posto 2** (é singular, $\det F = 0$).
- É definida **a menos de escala**, o que lhe dá **7 graus de liberdade** (9 elementos $-$ 1 de escala
  $-$ 1 da restrição de posto). Por isso pode ser estimada a partir de **correspondências de pontos** com
  o algoritmo dos 8 pontos (`cv2.findFundamentalMat`).
- Ela mapeia um ponto de uma imagem na sua **linha epipolar** na outra: $l_R = F\,x_L$.

Quando as câmeras estão **calibradas** (conhecemos os intrínsecos $K_L, K_R$), usa-se a **matriz
essencial** $E = K_R^{\top} F\,K_L$, que opera sobre coordenadas normalizadas e se decompõe diretamente
na **rotação $R$** e **translação $t$** entre as duas câmeras — os extrínsecos do par estéreo.


### 2.4 Disparidade estéreo

Depois de **retificar** o par (ver 2.5), as linhas epipolares ficam horizontais e um mesmo ponto do mundo
aparece na **mesma linha** nas duas imagens, deslocado apenas horizontalmente. Esse deslocamento é a
**disparidade**:

$$d = x_L - x_R.$$

A disparidade relaciona-se com a profundidade $Z$ por

$$Z = \frac{f \cdot B}{d},$$

onde $f$ é a distância focal (em pixels) e $B$ a *baseline*. A relação é **inversa**: disparidade grande
= objeto **próximo**; disparidade pequena = objeto **distante**; disparidade nula = ponto no infinito. O
**mapa de disparidade** (calculado, por exemplo, com `cv2.StereoBM`/`StereoSGBM`) é a base para o mapa de
profundidade explorado no laboratório seguinte.


### 2.5 Calibração estéreo e retificação

Calibrar o par estéreo envolve três etapas, todas presentes no programa `calibrate_abc.py` da equipe:

1. **Calibração individual (mono)** de cada câmera (`cv2.calibrateCamera`), obtendo $K_L, dist_L$ e
   $K_R, dist_R$ — exatamente como no Lab 4.
2. **Calibração estéreo** (`cv2.stereoCalibrate` com `CALIB_FIX_INTRINSIC`), que, mantendo os intrínsecos,
   estima a **rotação $R$** e a **translação $T$** entre as câmeras, além das matrizes **Essencial $E$** e
   **Fundamental $F$**. O módulo $\lVert T \rVert$ é a *baseline* em milímetros.
3. **Retificação** (`cv2.stereoRectify` + `cv2.initUndistortRectifyMap`), que gera os mapas que deixam as
   linhas epipolares **horizontais e alinhadas**, além da matriz de reprojeção **$Q$** (usada para levar
   disparidade a coordenadas 3D). Todos esses parâmetros são salvos em um arquivo **`.xml`**.


## 3. Construção da câmera estéreo (Parte 2)

Seguindo a referência [2] (*Making A Low-Cost Stereo Camera Using OpenCV*), a equipe montou a câmera
estéreo com **duas webcams USB iguais**, fixadas **paralelamente** sobre uma base rígida, com os eixos
ópticos paralelos e a *baseline* ajustada para ficar próxima à **distância interpupilar** humana. Após a
montagem, as câmeras foram **fixadas firmemente** para não se moverem uma em relação à outra — condição
essencial para que a calibração estéreo permaneça válida.

A **baseline efetivamente obtida** foi confirmada pela calibração (Seção 5.3): $\lVert T \rVert \approx
\mathbf{67{,}3\ mm}$, valor **compatível com a distância interpupilar humana** (tipicamente 60–70 mm), o
que valida o dimensionamento do arranjo.

> *Observação:* uma fotografia do arranjo físico das webcams pode ser inserida aqui como registro visual
> da montagem.


## 4. Ambiente e instruções de reprodução

Ambiente **Linux**, **Python 3** e **OpenCV** (o `requirements.txt` da pasta fixa
`opencv-contrib-python>=4.8`, `numpy>=1.24`, `tqdm>=4.65`). O padrão de calibração é o **mesmo tabuleiro
do Lab 4**, com **cantos internos $(8, 6)$** e quadrado de **25 mm** (escala métrica).

```bash
# 1) Ambiente virtual + dependencias
python3 -m venv venv && source venv/bin/activate
pip install -r requirements.txt

# 2) Capturar de 10 a 15 pares (L/R) do tabuleiro pelas duas webcams
python3 capture_images_abc.py            # tecla 'l' salva o par; 'q' sai

# 3) Calibrar a camera estereo (gera data/params_py.xml)
python3 calibrate_abc.py

# 4) Exibir ao vivo em 3D anaglifo
python3 movie3d_abc.py

# 5) Gravar o video 3D anaglifo (~15 s) e converter para mp4
python3 movie3d_abc_gravacao.py
```

> **Índices das câmeras.** No Linux as webcams aparecem como `/dev/video*`; muitas criam **dois** nós por
> dispositivo, por isso a câmera direita usa o índice `2`. Ajuste `CamL_id`/`CamR_id` conforme
> `v4l2-ctl --list-devices`. Os dados desta execução estão na subpasta `data/`.


## 5. Procedimentos experimentais (Parte 3)

### 5.1 (A) Obtenção dos códigos de exemplo

Os códigos de exemplo da câmera estéreo da LearnOpenCV (referência [2], repositório `stereo-camera`)
foram baixados e estão nesta pasta: `capture_images.py`, `calibrate.py`, `movie3d.py` (e as versões em
C++ `capture_images.cpp`, `calibrate.cpp`, `movie3d.cpp`), além das subpastas de imagens **L** (esquerda)
e **R** (direita). Esses exemplos serviram de base para os programas adaptados da equipe
(`*_abc.py`). Conforme o roteiro, os arquivos baixados no computador do laboratório foram apagados ao
final da aula; aqui mantemos apenas cópias necessárias à reprodução.


### 5.2 (B) Execução do exemplo com as imagens fornecidas

O exemplo foi executado com as imagens fornecidas (subpastas `data/stereoL/img*.png` e
`data/stereoR/img*.png`) para validar o *pipeline* antes de usar a câmera própria; o resultado de
referência acompanha o exemplo em `data/params_cpp.xml`.

**Parâmetros necessários para uma câmera estéreo** (resposta ao roteiro): para cada câmera, os
**intrínsecos** $K_L, K_R$ e os coeficientes de **distorção** $dist_L, dist_R$; e, para o par, a
**rotação $R$** e a **translação $T$** entre as câmeras, as matrizes **Essencial $E$** e **Fundamental
$F$**, os **mapas de retificação** e a matriz de reprojeção **$Q$**. **Há arquivo necessário?** Sim: a
calibração produz um arquivo de parâmetros (`.xml`) que precisa ser carregado pelo programa de exibição
3D (`movie3d`) — sem ele não é possível retificar as imagens ao vivo.


### 5.3 (C) Calibração da câmera estéreo construída

A equipe capturou **15 pares** de imagens do tabuleiro com a câmera estéreo (`capture_images_abc.py`,
arquivos `data/stereoL/lucas*.png` e `data/stereoR/lucas*.png`, nomeados com o integrante Lucas conforme
o roteiro) e executou `calibrate_abc.py`. Os **15 pares foram válidos** (tabuleiro detectado nas duas
câmeras). Os parâmetros foram salvos em `data/params_py.xml`. A célula abaixo relê o arquivo e imprime os
parâmetros obtidos.


In [ ]:
import cv2, numpy as np
np.set_printoptions(precision=4, suppress=True)

fs = cv2.FileStorage("data/params_py.xml", cv2.FILE_STORAGE_READ)
for chave in ["mtxL", "distL", "mtxR", "distR", "R", "T", "E", "F", "Q", "baseline_mm"]:
    no = fs.getNode(chave)
    if chave == "baseline_mm":
        print("baseline_mm =", no.real())
    else:
        print(chave, "=\n", no.mat())
fs.release()


**Parâmetros obtidos pela equipe** (de `data/params_py.xml`).

Intrínsecos e distorção de cada câmera:

$$K_L = \begin{bmatrix} 682{,}32 & 0 & 336{,}02 \\ 0 & 681{,}28 & 224{,}44 \\ 0 & 0 & 1 \end{bmatrix},\quad
\texttt{dist}_L = [\,0{,}0126,\ 0{,}0889,\ -0{,}0066,\ 0{,}0033,\ 0{,}1194\,]$$

$$K_R = \begin{bmatrix} 686{,}62 & 0 & 343{,}98 \\ 0 & 685{,}23 & 221{,}64 \\ 0 & 0 & 1 \end{bmatrix},\quad
\texttt{dist}_R = [\,-0{,}0107,\ 0{,}3763,\ -0{,}0068,\ 0{,}0043,\ -1{,}1237\,]$$

Extrínsecos do par (rotação $R$ e translação $T$, em mm):

$$R = \begin{bmatrix} 0{,}9999 & -0{,}0052 & 0{,}0122 \\ 0{,}0050 & 0{,}9998 & 0{,}0167 \\ -0{,}0123 & -0{,}0166 & 0{,}9998 \end{bmatrix},\qquad
T = \begin{bmatrix} -65{,}79 \\ 3{,}53 \\ -13{,}72 \end{bmatrix}\ \text{mm}$$

Matriz **Essencial**, **Fundamental** e de **reprojeção $Q$**:

$$E = \begin{bmatrix} 0{,}02 & 13{,}66 & 3{,}76 \\ -14{,}53 & -1{,}02 & 65{,}60 \\ -3{,}86 & -65{,}76 & -1{,}14 \end{bmatrix},\quad
F = \begin{bmatrix} 0 & 0 & 0{,}001 \\ 0 & 0 & -0{,}105 \\ -0{,}001 & 0{,}102 & 1 \end{bmatrix}$$

$$Q = \begin{bmatrix} 1 & 0 & 0 & -171{,}43 \\ 0 & 1 & 0 & -212{,}05 \\ 0 & 0 & 0 & 676{,}57 \\ 0 & 0 & 0{,}0149 & 0 \end{bmatrix}$$

**Baseline:** $\lVert T \rVert = \mathbf{67{,}29\ mm}$.

**Erro de reprojeção (RMS)** obtido na calibração dos 15 pares: câmera esquerda **0,131 px**, câmera
direita **0,130 px** e **estéreo 1,003 px** — valores baixos, indicando uma calibração de boa qualidade
(mono bem abaixo de 1 px; o RMS estéreo pouco acima de 1 px é típico por acoplar as duas câmeras).

**Interpretação.** A matriz $R$ é praticamente a **identidade** (rotação relativa de ~1°), confirmando
que as webcams ficaram bem **paralelas**. O vetor $T$ é dominado pela componente $x$ ($-65{,}8$ mm), ou
seja, o deslocamento é essencialmente **lateral** (a *baseline*), com pequenas componentes em $y$ e $z$
por desalinhamento residual. Na matriz $Q$, o elemento $Q_{2,3}=676{,}57$ corresponde à **distância
focal** média em pixels e $Q_{3,2}=0{,}0149 \approx 1/67{,}3$ codifica o inverso da *baseline*.

**Quais parâmetros foram salvos no `.xml`? (pergunta do roteiro).** O `calibrate_abc.py` grava em
`data/params_py.xml`: os **mapas de retificação** (`Left_Stereo_Map_x/y`, `Right_Stereo_Map_x/y`), os
**intrínsecos** `mtxL`, `mtxR`, as **distorções** `distL`, `distR`, os **extrínsecos** `R`, `T`, as
matrizes `E`, `F`, a matriz de reprojeção `Q`, a `baseline_mm` e o tamanho da imagem
(`image_width`, `image_height` = 640×480).

**Evidência da retificação.** Aplicando os mapas de retificação a um par capturado (`lucas1`), as linhas
epipolares tornam-se **horizontais e coincidentes** entre as duas imagens: os cantos correspondentes do
tabuleiro caem sobre as **mesmas linhas verdes** nas vistas esquerda e direita.

![Par retificado (linhas epipolares horizontais alinhadas)](data/retificacao_lucas1.png)

*Figura 1 — Par `lucas1` após retificação (`cv2.remap` com os mapas de `params_py.xml`). As linhas verdes
horizontais cruzam os mesmos pontos nas duas imagens, evidenciando o alinhamento epipolar.*


### 5.4 (D) Gravação de um vídeo 3D anaglifo

A partir do programa de exibição ao vivo (`movie3d_abc.py`), a equipe elaborou o
`movie3d_abc_gravacao.py`, que **grava** um vídeo 3D **anaglifo** (vermelho/ciano) de ~15 s e o converte
para **mp4**. Ambos leem os mapas de retificação de `data/params_py.xml`, retificam cada quadro das duas
webcams e montam o anaglifo colocando o **canal vermelho** da câmera **esquerda** sobre os **canais
verde+azul** da câmera **direita**. Trechos essenciais:


In [ ]:
# movie3d_abc.py (nucleo) — anaglifo AO VIVO a partir das duas webcams.
import cv2

# Le os mapas de retificacao salvos pela calibracao.
fs = cv2.FileStorage("data/params_py.xml", cv2.FILE_STORAGE_READ)
Lx = fs.getNode("Left_Stereo_Map_x").mat();  Ly = fs.getNode("Left_Stereo_Map_y").mat()
Rx = fs.getNode("Right_Stereo_Map_x").mat(); Ry = fs.getNode("Right_Stereo_Map_y").mat()
fs.release()

CamL = cv2.VideoCapture(0, cv2.CAP_V4L2)
CamR = cv2.VideoCapture(2, cv2.CAP_V4L2)

while True:
    retL, imgL = CamL.read(); retR, imgR = CamR.read()
    if not (retL and retR):
        break
    # Retifica + corrige distorcao das duas imagens.
    Left_nice  = cv2.remap(imgL, Lx, Ly, cv2.INTER_LANCZOS4, cv2.BORDER_CONSTANT, 0)
    Right_nice = cv2.remap(imgR, Rx, Ry, cv2.INTER_LANCZOS4, cv2.BORDER_CONSTANT, 0)
    # Anaglifo (BGR): R (indice 2) da ESQUERDA; G e B da DIREITA.
    output = Right_nice.copy()
    output[:, :, 2] = Left_nice[:, :, 2]
    cv2.imshow("3D movie", cv2.resize(output, (700, 700)))
    if (cv2.waitKey(1) & 0xFF) == ord("q"):
        break
CamL.release(); CamR.release(); cv2.destroyAllWindows()


In [ ]:
# movie3d_abc_gravacao.py (nucleo) — grava ~15 s e converte para mp4.
import time, subprocess, cv2

DURACAO_S, FPS, TAM = 15, 20, (700, 700)
writer = cv2.VideoWriter("data/anaglifo.avi",
                         cv2.VideoWriter_fourcc(*"MJPG"), FPS, TAM)
t0 = time.time()
while True:
    retL, imgL = CamL.read(); retR, imgR = CamR.read()
    if not (retL and retR):
        break
    Left_nice  = cv2.remap(imgL, Lx, Ly, cv2.INTER_LANCZOS4, cv2.BORDER_CONSTANT, 0)
    Right_nice = cv2.remap(imgR, Rx, Ry, cv2.INTER_LANCZOS4, cv2.BORDER_CONSTANT, 0)
    output = Right_nice.copy()
    output[:, :, 2] = Left_nice[:, :, 2]     # vermelho da esquerda
    writer.write(cv2.resize(output, TAM))    # grava o quadro
    if (cv2.waitKey(1) & 0xFF) == ord("q") or time.time() - t0 >= DURACAO_S:
        break
writer.release()
# Converte AVI -> MP4 (item D do roteiro):
subprocess.run(["ffmpeg", "-y", "-i", "data/anaglifo.avi",
                "-c:v", "libx264", "-pix_fmt", "yuv420p", "data/anaglifo.mp4"])


**Resultado obtido pela equipe.** Foi gravado um vídeo anaglifo de **~18,7 s** (374 quadros a 20 FPS,
resolução 700×700), salvo em `data/anaglifo.avi` e convertido para `data/anaglifo.mp4`. A figura abaixo é
um **quadro** desse vídeo: o deslocamento **vermelho/ciano** entre as duas vistas (nítido nas bordas do
tabuleiro e nos contornos da pessoa) é justamente a **disparidade** que, vista com os óculos anaglifo,
produz a sensação de profundidade.

![Quadro do vídeo 3D anaglifo gravado pela equipe](data/anaglifo_frame.png)

*Figura 2 — Quadro de `data/anaglifo.mp4`: anaglifo vermelho/ciano (canal vermelho da câmera esquerda +
canais verde/azul da direita). Use óculos com lente vermelha à esquerda e ciano à direita.*

**Vídeo completo:** [`data/anaglifo.mp4`](data/anaglifo.mp4) (versão `.avi` original em
`data/anaglifo.avi`).

<video src="data/anaglifo.mp4" controls width="480"></video>

**Percepção da equipe (item D do roteiro).** Com os óculos anaglifo, o efeito de profundidade é claramente
percebido — o tabuleiro "salta" à frente do fundo. *Cada integrante deve registrar aqui sua percepção
individual* e a comparação entre a experiência **ao vivo** (`movie3d_abc.py`) e a do **vídeo gravado**;
tipicamente a versão ao vivo tem maior sensação de imersão (sem compressão), enquanto o vídeo gravado
facilita a análise por permitir pausar e repetir.


### 5.5 (E) Aplicação da câmera estéreo no Trabalho Final

O Trabalho Final da equipe é a **Detecção de Uso de EPI em Ambiente Simples**. Embora o protótipo principal
use uma única câmera, a **câmera estéreo construída aqui agrega valor direto** ao projeto:

- **Filtro por profundidade / zona de interesse.** Com a matriz $Q$ e a *baseline* calibrada é possível
  estimar a **distância** de cada pessoa à câmera e só avaliar EPI de quem está dentro da **zona de
  acesso** (p. ex., 1,5–3 m), ignorando pessoas ao fundo.
- **Redução de falsos positivos.** A profundidade distingue uma **pessoa real** de imagens **planas**
  (cartazes, telas, fotos de trabalhadores com capacete), um erro comum em detectores monoculares.
- **Medição de tamanho real.** A escala métrica (quadrado de 25 mm) permite estimar **dimensões reais** —
  útil, por exemplo, para confirmar que o objeto na cabeça tem o tamanho de um capacete.
- **Robustez a oclusão/pose.** Duas vistas fornecem informação redundante, ajudando quando um EPI fica
  parcialmente oculto em uma das câmeras.

Assim, a mesma montagem e o mesmo procedimento de calibração estéreo deste laboratório podem ser
**reaproveitados** como um módulo opcional de profundidade no sistema de EPI.


## 6. Análise e discussão

**Qualidade da calibração.** Os erros de reprojeção foram **baixos**: 0,131 px (esquerda) e 0,130 px
(direita) na calibração mono, e 1,003 px na calibração estéreo. Isso indica que a detecção de cantos e a
estimativa dos parâmetros foram precisas, e que o par de webcams está bem descrito pelo modelo.

**Geometria do arranjo.** A rotação relativa $R \approx I$ (≈1°) e a translação dominada pela componente
lateral confirmam que as câmeras ficaram **paralelas e bem fixadas**, como exige o método. A **baseline**
de **67,3 mm** ficou dentro da faixa da **distância interpupilar** humana (60–70 mm), atendendo ao
requisito do roteiro e garantindo uma disparidade natural para a percepção 3D.

**Retificação e efeito 3D.** A Figura 1 comprova a retificação — cantos correspondentes sobre as mesmas
linhas horizontais —, o que é pré-requisito para o anaglifo e para futuros mapas de disparidade
(Lab 6). O vídeo anaglifo (Figura 2) apresentou deslocamento vermelho/ciano coerente e sensação de
profundidade nítida com os óculos.

**Dificuldades encontradas e soluções.** (i) **Índices das câmeras** no Linux — webcams criam vários nós
`/dev/video*`; resolvido fixando `CamL_id=0`/`CamR_id=2` e o backend `CAP_V4L2`. (ii) **Sincronismo e
fixação** — qualquer movimento relativo entre as webcams invalida a calibração; resolvido fixando-as
rigidamente antes de calibrar. (iii) **Captura de bons pares** — o programa de captura só salva o par
quando o tabuleiro é detectado **nas duas** câmeras simultaneamente, o que garantiu os 15 pares válidos.

**Limitações.** A resolução 640×480 e a iluminação afetam a densidade de disparidade; para mapas de
profundidade densos (Lab 6) convém mais pares de calibração e controle de exposição das duas câmeras.


## 7. Conclusões

O Laboratório 5 cumpriu seus objetivos. A equipe estudou a **geometria epipolar** — epipolos, plano e
linha epipolar, matrizes fundamental e essencial e o conceito de disparidade —, **construiu** uma câmera
estereoscópica de baixo custo com duas webcams e a **calibrou**, obtendo todos os parâmetros do par:
intrínsecos e distorção de cada câmera, os extrínsecos $R$ e $T$, as matrizes $E$, $F$ e $Q$ e os mapas
de retificação.

Os resultados confirmaram a qualidade da montagem: erros de reprojeção de **0,131 px** e **0,130 px** nas
calibrações mono e **1,003 px** na estéreo, rotação relativa praticamente identidade (≈1°, câmeras bem
paralelas) e *baseline* de **67,3 mm**, dentro da faixa da distância interpupilar humana, como pedia o
roteiro. A retificação foi comprovada visualmente pelo alinhamento dos cantos do tabuleiro sobre as mesmas
linhas horizontais nas duas vistas.

Por fim, o vídeo **anaglifo** de ~18,7 s demonstrou o efeito estereoscópico na prática, com percepção de
profundidade nítida ao ser observado com os óculos vermelho-ciano. As principais dificuldades —
identificação dos índices das webcams no Linux, fixação rígida do arranjo e captura de pares válidos —
foram contornadas com o backend V4L2, a fixação prévia à calibração e a validação automática do tabuleiro
nas duas câmeras. Os parâmetros obtidos servem de base direta para o mapa de profundidade do Laboratório 6.


## 8. Declaração de uso de Inteligência Artificial Generativa

Em atendimento à **Portaria CNPq nº 2664/2026**, a equipe declara ter utilizado um assistente de IA
generativa (Anthropic — Claude) apenas como apoio à **formatação e organização textual** deste
documento — estruturação das seções, revisão de escrita e formatação do notebook. Todo o conteúdo
técnico, os dados e os resultados são de autoria da equipe, que é integralmente responsável pelo
material final.


## 9. Referências

1. LearnOpenCV — *Introduction to Epipolar Geometry and Stereo Vision*. Disponível em:
   https://learnopencv.com/introduction-to-epipolar-geometry-and-stereo-vision/
2. LearnOpenCV — *Making A Low-Cost Stereo Camera Using OpenCV*. Disponível em:
   https://learnopencv.com/making-a-low-cost-stereo-camera-using-opencv/
   (código: https://github.com/spmallick/learnopencv/tree/master/stereo-camera)
3. LearnOpenCV — *Understanding Lens Distortion*. Disponível em:
   https://learnopencv.com/understanding-lens-distortion/
4. LOOP, C.; ZHANG, Z. *Computing Rectifying Homographies for Stereo Vision*. IEEE CVPR, 1999.
5. LearnOpenCV — *Geometry of Image Formation*. Disponível em:
   https://learnopencv.com/geometry-of-image-formation/
6. CNPq — *Portaria nº 2664/2026* (diretrizes de integridade e uso de IAG). Disponível em:
   http://memoria2.cnpq.br/web/guest/view/-/journal_content/56_INSTANCE_0oED/10157/23142775
